# FMQA

## Intro
**FMQA**, short for **Factorization Machine (FM)** with **Quantum Annealing (QA)**, is a method designed to solve combinatorial optimization problems. In this approach, the physical structure is represented in binary form, and we refer to this physical structure as a "configuration". A Figure of Merit (FoM) is then defined to evaluate its performance. The goal of FMQA is to identify the configuration that minimizes the FoM.  

Due to the hardware limitations of QA, it can only solve problems in Ising form (Eq. 1) — or equivalently, in QUBO form (Eq. 2). However, designing a FoM directly in QUBO form is not an easy task; in most cases, we can only formulate the FoM in PUBO form (Eq. 3). Since FM (Eq. 4) and QUBO form differ only by a constant term, which makes no difference in optimization problems, we can use FM to fit a QUBO form (surrogate FoM) from the PUBO form (original FoM). QA can then optimize the surrogate FoM.

$$
\text{Ising}: \quad
E(\boldsymbol{s}) = \sum_{i=1}^{N} h_i s_i + \sum_{i=1}^{N}\sum_{j=i+1}^{N} J_{i,j} s_i s_j, \quad\boldsymbol{s} = \{-1, 1\}^{N}
\tag{1}
$$

$$
\text{QUBO}: \quad
f(\boldsymbol{x}) = \sum_{i=1}^{N} Q_i x_i + \sum_{i=1}^{N}\sum_{j=i+1}^{N} Q_{i,j} x_i x_j, \quad\boldsymbol{x} = \{0, 1\}^{N}
\tag{2}
$$

$$
\text{PUBO}: \quad
f(\boldsymbol{x}) = \sum_{i=1}^{N} Q_i x_i + \sum_{i=1}^{N}\sum_{j=i+1}^{N} Q_{i,j} x_i x_j + \sum_{i=1}^{N}\sum_{j=i+1}^{N}\sum_{k=j+1}^{N} Q_{i,j,k} x_i x_j x_k + ..., \quad\boldsymbol{x} = \{0, 1\}^{N}
\tag{3}
$$

$$
\text{FM}: \quad
\hat{y}(\boldsymbol{x}) = w_0 + \sum_{i=1}^{N} w_i x_i + \sum_{i=1}^{N}\sum_{j=i+1}^{N} w_{i,j} x_i x_j
\tag{4}
$$

The coupling terms $(w_{i,j})$ in FM have some constraints. During training, FM in fact learns a matrix $\boldsymbol{V} \in \mathbb{R}^{N \times K}$. We can regard $\mathbf{V}$ as consisting of $N$ vectors, where each vector $\boldsymbol{v_i} \in \mathbb{R}^K$. The coupling term defined in FM is obtained by taking the inner product between these $K$-dimensional vectors (Eq. 5).  
Note. $K$ is a hyperparameter

$$
w_{i,j} = \boldsymbol{v_i} \cdot \boldsymbol{v_j} = \sum_{f=1}^{K} v_{i,f} v_{j,f}
\tag{5}
$$

Naturally, fitting a PUBO form with a QUBO form is impossible. To improve its accuracy in the regions that matter most, We first use QA to sample low-energy states of the QUBO form. These data are then added back into the FM training set, effectively enriching it with more critical samples. By repeating this process, the training set gradually accumulates more data in the low-energy, allowing the QUBO form to more closely approximate the PUBO form in local region.  

<img src="./fmqa_flowchart.png" width="400" height="250" style="display:block; margin:auto;">
<br>

Note. From our tests, we observed that certain FoMs do not decrease as expected. The underlying reason remains unclear, and at this stage, we can only proceed through trial and error.  
Note. If the FoM is not able to reliably evaluate the performance of configurations, post-processing becomes necessary to extract the truly good configurations.  

For more theoretical details, please refer to the following paper.  
FM: [Rendle, Proc. ICDM, IEEE, pp. 995–1000 (2010).](https://ieeexplore.ieee.org/document/5694074)  
FMQA: [Kitai et al., Phys. Rev. Research 2, 013319 (2020).](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.2.013319)

## Physical problem (superoscillation with distant sidebands)
Our main purpose is to optimize the configuration of the **Binary Phase Zone Plate (BPZP)**, enabling it to achieve superoscillatory focusing while pushing the sidebands farther away.  
This program has been modularized, and all BPZP-related functionalities have been implemented in ***zone_plate.py***. Therefore, you can easily adapt the program to your own physical problem.  

## Implementation

Before diving into the practical part, let me introduce some terms that will be used later.
- **Binary Quadratic Model (BQM)**: A class defined by D-Wave's package which is equivalent to Ising form and QUBO form (there is an parameter to set which form you use)
- **Simulated Annealing (SA)**: A classical heuristic algorithm which can solve QUBO form problem as well (QA compatible)

The FMQA implementation was provided by Kitai on [GitHub](https://github.com/tsudalab/fmqa). However, we made some adjustments for our own purposes.  
- We only used the FM part of the program and did not use the QA part, since our tests showed that the BQM behaved differently than we expected and could not be run on a quantum annealer
- We integrated various optimization algorithms provided by D-Wave, allowing an easy switch between QA and SA

Note. In our practical use of QA, we found that under certain unknown conditions there can be an high rate of broken chains, making the optimization results unreliable. Based on our tests, the optimization results are only reliable when the proportion of broken chains remains below 10%.  

<br>

We divide the program into several parts:
- **Import**: Import the required packages
- **Factorization Machine**: Extend the original FM with additional functionalities
- **Variables & Objects**: Set initial values and initialize objects
- **Paths**: Define the necessary paths
- **Funcs**: Implement the functions required for FMQA
- **FMQA**: Execute FMQA

## Convert to a script
If you want to run the script file by using **run_fmqa.ipynb**, there are some code need to change  
```
step_pt of target: float(sys.argv[1])
max_val of target: float(sys.argv[2])
if len(sys.argv) > 3:
    program_id = sys.argv[3]
else:
    program_id = None
```

## Import

In [1]:
import os
import sys
import time
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
from zone_plate import ZonePlate

# Setting Environment Variable (Has to be set before importing mxnet, or it will perform multiprocessing automatically but without any imrpoved efficiency)
os.environ['MXNET_CPU_WORKER_NTHREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'

# Factorization Machine
import mxnet as mx
from mxnet import nd
from factorization_machine import FactorizationMachine as OriginalFactorizationMachine

# Fix the random seeds
random.seed(int(time.time()))
np.random.seed(int(time.time()))
mx.random.seed(int(time.time()))

# QUBO Sampler
import dimod
import dwave.inspector
from dimod import ExactSolver
from dwave.samplers import RandomSampler, SimulatedAnnealingSampler, SteepestDescentSampler, TabuSampler
from dwave.system import DWaveSampler, DWaveCliqueSampler, EmbeddingComposite, FixedEmbeddingComposite

## Factorization Machine

In [2]:
class FactorizationMachine(OriginalFactorizationMachine):

    def __call__(self, xs):
        """Override __call__ to handle numpy array input automatically."""
        if isinstance(xs, np.ndarray):
            xs = nd.array(xs)    # Convert numpy array to MXNet NDArray
        return super().__call__(xs)
    
    def loss(self, dataset):
        """
        Compute loss
        - mean((ys - outputs)**2)
    
        Parameters
        - dataset: [xs, ys]
        """
        xs = nd.array(dataset[0])
        ys = nd.array(dataset[1])
        return nd.mean( ( ys - self(xs) ) ** 2 ).asscalar()

    def get_bhQ_scaled(self):
        """Apply scaling to the BQM"""
        # b: bias, h: linear, Q: quadratic (upper triangular part), all as numpy arrays
        b, h, Q = self.get_bhQ()  
        
        h_max = np.max(np.abs(h))
        Q_max = np.max(np.abs(Q))
        scaling_factor = max(h_max, Q_max)
        
        b /= scaling_factor
        h /= scaling_factor
        Q /= scaling_factor

        return b, h, Q

    def bqm(self):
        """Generate BQM from the model"""
        b, h, Q = self.get_bhQ()
        # b, h, Q = self.get_bhQ_scaled()    # b: bias, h: linear, Q: quadratic (upper triangular part), all as numpy arrays

        return dimod.BinaryQuadraticModel(h, Q, b, dimod.BINARY)    # BQM(linear, quadratic, offset, vartype)

    def plot_Q_matrix(self, show_fig=False, path_fig="Q_matrix_heatmap.png", save_fig=True, path_data="Q_matrix.npy", save_data=True):
        """Plot heatmap of the Q matrix"""
        b, h, Q = self.get_bhQ()
        # b, h, Q = self.get_bhQ_scaled()    # bias, linear, qudratic(upper tri)
        Q = Q + Q.T
        np.fill_diagonal(Q, h)
        
        if save_data:
            np.save(path_data, Q)

        if show_fig or save_fig:
            max_abs_val = np.max(np.abs(Q))
            plt.figure()
            plt.imshow(Q, cmap="bwr", vmin=-max_abs_val, vmax=max_abs_val)
            plt.colorbar()
            plt.title("Q Matrix")

            if save_fig:
                plt.savefig(path_fig, bbox_inches='tight', transparent=True)
            if show_fig:
                plt.show()
            else:
                plt.close()

    def save_model(self, path="model.params"):
        """Save model parameters and optimizer states"""
        mx.nd.waitall()
        self.save_parameters(path)
        
        if self.trainer is not None:
            self.trainer.save_states(path + ".trainer")

    @staticmethod
    def load_model(var_num, K, path="model.params"):
        """
        Load and return model parameters and optimizer states

        Parameters
        - var_num : number of qubits
        - K       : factorization size of the Factorization Machine
        """
        model = FactorizationMachine(input_size=var_num, factorization_size=K, act="identity")
        model.load_parameters(path, ctx=mx.cpu())

        # Trainer must be initialized before loading its state
        model.trainer = mx.gluon.Trainer(model.collect_params(), "adam")

        try:
            model.trainer.load_states(path + ".trainer")
        except FileNotFoundError:
            print("Optimizer state not found")

        return model

## Variables & Objects

In [3]:
# vars & obj for ZonePlate
rings      = 16
multiple   = 8.0
resolution = 2_000
r_vec      = np.linspace(0, multiple, resolution)    # unit: wavelength
wavelength = 0.53                                    # unit: µm
k          = 2 * np.pi / wavelength
NA         = 0.95

zp = ZonePlate(multiple, resolution, wavelength, NA)

target_method = "intensity_step(squared)"
target_max_mul = multiple
target_step_point_mul = 0.3
target_step_left_val = 0.1
target = [target_method, target_max_mul, target_step_point_mul, target_step_left_val]
target_str = "_".join(map(str, target))

In [4]:
# vars for FM
input_size    = rings    # dimension of each config
K             = rings    # the latent factor in FM
num_epoch     = 1_000
learning_rate = 1.0e-2

In [5]:
# vars for QUBO
sampler_type = "SA"    # Exact, Random, SA, SD, Tabu, SA_SD, SA_Tabu, QA_Auto, QA_Clique
qpu_solver = None      # {"name": "Advantage2_system1.1"}

In [6]:
# vars for FMQA
init_dataset_size = 100
iteration         = 5
num_reads         = 100         # annealing times per iter
adding_method     = "energy"    # "energy" | "FoM"
adding_num        = 100

## Paths

In [7]:
def create_init_path(base_folder_name, program_id=None):
    """Creates a folder for this simulation

    Args:
        base_folder_name (str): Name of the parent directory (must be a valid path string).
        program_id (int, optional): Unique identifier for the simulation task. Defaults to None (auto-generated).
    """
    os.makedirs(base_folder_name, exist_ok=True)

    if program_id is None:
        # Find the maximum value among all numeric folders
        existing_nums = []
        for folder in os.listdir(base_folder_name):
            folder_path = os.path.join(base_folder_name, folder)
            if os.path.isdir(folder_path) and folder.isdigit():
                existing_nums.append(int(folder))
        
        # Set program_id
        program_id = str(max(existing_nums, default=0) + 1)
    else:
        # Ensure program_id is a string
        program_id = str(program_id)

    # Creates a folder corresponding to a specific program_id
    base_path = os.path.join(base_folder_name, program_id)
    os.makedirs(base_path)
    
    return program_id

In [8]:
# Set base path
base_folder_name = "saved_files"
program_id       = create_init_path(base_folder_name, program_id=None)
base_path        = os.path.join(base_folder_name, program_id)
print(f"program id: {program_id}")

# Path for saving the parameter settings file
params_file_name = "params.txt"
params_path      = os.path.join(base_path, params_file_name)

# Base path for configs & foms 
# (saved in a configs_foms folder at the same level as the script, so other programs can also use it)
configs_foms_folder_name = "configs_foms"
os.makedirs(configs_foms_folder_name, exist_ok=True)

# File names for the initial generated configurations & their corresponding FoMs
init_configs_file_name = f"init_configs_{init_dataset_size}_{rings}_{multiple}_{resolution}_{wavelength}_{NA}.npy"
init_foms_file_name    = f"init_foms_{init_dataset_size}_{target_str}_{rings}_{multiple}_{resolution}_{wavelength}_{NA}.npy"
init_configs_path      = os.path.join(configs_foms_folder_name, init_configs_file_name)
init_foms_path         = os.path.join(configs_foms_folder_name, init_foms_file_name)

# Configs, energies, and FoMs obtained after running FMQA
configs_file_name  = "configs.npy"
energies_file_name = "energies.npy"
foms_file_name     = "foms.npy"
configs_path       = os.path.join(base_path, configs_file_name)
energies_path      = os.path.join(base_path, energies_file_name)
foms_path          = os.path.join(base_path, foms_file_name)

# The new data added in each FMQA iteration (actual_new_data)
actual_new_data_file_name = "actual_new_data.npy"
actual_new_data_path = os.path.join(base_path, actual_new_data_file_name)

# Model and Q_matrix for each FMQA iteration 
# (path needs to be adjusted for every iteration)
iter_model_Q_folder_name = "model&Q"
iter_model_Q_path        = os.path.join(base_path, iter_model_Q_folder_name)
os.makedirs(iter_model_Q_path)
iter_model_file_name = "model.params"
iter_Q_arr_file_name = "Q.npy"
iter_Q_fig_file_name = "Q.png"

program id: 1118


In [9]:
# write the setting vars to a file

variables = {
    
    # vars for ZonePlate
    "rings"     : rings,
    "multiple"  : multiple,
    "resolution": resolution,
    "wavelength": wavelength,
    "NA"        : NA,
    "target"    : target,
    
    # vars for FM
    "K"        : K,
    "num_epoch": num_epoch,
    
    # vars for QUBO
    "sampler_type": sampler_type,
    "qpu_solver"  : qpu_solver,
    
    # vars for FMQA
    "init_dataset_size": init_dataset_size,
    "iteration"        : iteration,
    "num_reads"        : num_reads,
    "adding_method"    : adding_method,
    "adding_num"       : adding_num,
}

with open(params_path, "w", encoding="utf-8") as f:
    for key, value in variables.items():
        f.write(f"{key} = {value}\n")

## Funcs

In [10]:
def generate_unique_binary_array(init_dataset_size, rings):
    max_unique = 2 ** rings
    
    if init_dataset_size > max_unique:
        raise ValueError(f"Cannot generate {init_dataset_size} unique rows with only {rings} bits. Maximum is {max_unique}.")
    
    def unique_binary_generator():
        seen = set()
        while len(seen) < init_dataset_size:
            new_row = tuple(np.random.randint(0, 2, size=rings, dtype=np.int8))
            if new_row not in seen:
                seen.add(new_row)
                yield np.array(new_row, dtype=np.int8)
    
    # Use a generator to yield rows one at a time
    result = np.array(list(unique_binary_generator()))
    
    # Shuffle the rows
    np.random.shuffle(result)
    
    return result

In [11]:
def choose_sampler(sampler_type = "SA"):
    """
    Define which sampler to use.
    Global variable: qpu_solver
    """
    if sampler_type == "Exact":
        if rings <= 16:
            sampler = ExactSolver()
        else:
            print("Too many rings, Exact Solver cannot be used")
            sys.exit(1)
    elif sampler_type == "Random":
        sampler = RandomSampler()
    elif sampler_type == "SA":
        sampler = SimulatedAnnealingSampler()
    elif sampler_type == "SD":
        sampler = SteepestDescentSampler()
    elif sampler_type == "Tabu":
        sampler = TabuSampler()
    elif sampler_type == "QA_Auto":
        sampler = EmbeddingComposite(DWaveSampler(qpu_solver))
    elif sampler_type == "QA_Clique":
        sampler = DWaveCliqueSampler(qpu_solver)
    else:
        print("No such QUBO sampler")
        sys.exit(1)
    
    return sampler

def sampling(sampler_type, bqm, num_reads=None):
    """
    Start annealing/sampling.
    Global variables: None
    """

    sampler_types = ["Exact", "Random", "SA", "SD", "Tabu", "QA_Auto", "QA_Clique"]
    if sampler_type in sampler_types:
        sampler = choose_sampler(sampler_type)
    
    # Single simulation
    if sampler_type == "Exact":
        sampleset = sampler.sample(bqm)
    elif sampler_type == "Random":
        sampleset = sampler.sample(bqm, num_reads=num_reads)
    elif sampler_type == "SA":
        sampleset = sampler.sample(bqm, num_reads=num_reads, num_sweeps=1_000)
    elif sampler_type == "SD":
        sampleset = sampler.sample(bqm, num_reads=num_reads)
    elif sampler_type == "Tabu":
        sampleset = sampler.sample(bqm, num_reads=num_reads)
    elif sampler_type == "QA_Auto":
        sampleset = sampler.sample(bqm, num_reads=num_reads, annealing_time=20)
    elif sampler_type == "QA_Clique":
        sampleset = sampler.sample(bqm, num_reads=num_reads, annealing_time=20)

    # Multiple simulations
    elif sampler_type == "SA_SD":
        sampler = choose_sampler("SA")
        sampleset = sampler.sample(bqm, num_reads=num_reads, num_sweeps=1_000)
        sampler = choose_sampler("SD")
        sampleset = sampler.sample(bqm, num_reads=num_reads, initial_states=sampleset)
    elif sampler_type == "SA_Tabu":
        sampler = choose_sampler("SA")
        sampleset = sampler.sample(bqm, num_reads=num_reads, num_sweeps=1_000)
        sampler = choose_sampler("Tabu")
        sampleset = sampler.sample(bqm, num_reads=num_reads, initial_states=sampleset)

    # 
    else:
        print("No such sampler")
        sys.exit(1)
    
    return sampleset

In [12]:
def add_data(configs, energies, foms, sample_configs, sample_energies, adding_method, adding_num):
    
    # step1: choose
    if adding_method == "energy":
        sorted_indices  = np.argsort(sample_energies)[:adding_num]
        sample_configs  = sample_configs[sorted_indices]
        sample_energies = sample_energies[sorted_indices]
        sample_foms     = np.array([
                              zp.set_config(config) or zp.figure_of_merit_calc.calc_fom(target)
                              for config in sample_configs
                          ])
    elif adding_method == "FoM":
        sample_foms     = np.array([
                              zp.set_config(config) or zp.figure_of_merit_calc.calc_fom(target)
                              for config in sample_configs
                          ])
        sorted_indices  = np.argsort(sample_foms)[:adding_num]
        sample_configs  = sample_configs[sorted_indices]
        sample_energies = sample_energies[sorted_indices]
        sample_foms     = sample_foms[sorted_indices]
    else:
        return
    
    # step2: filter out the repeated data
    unique_indices  = np.unique(sample_configs, axis=0, return_index=True)[1]    # Remove duplicate rows in sample_configs and drop corresponding elements in sample_foms
    unique_indices  = np.sort(unique_indices)
    sample_configs  = sample_configs[unique_indices]
    sample_energies = sample_energies[unique_indices]
    sample_foms     = sample_foms[unique_indices]
    mask = np.array([not np.any(np.all(configs == row, axis=1)) for row in sample_configs])    # Remove rows in sample_configs that already appear in configs, and drop corresponding elements in sample_foms
    sample_configs  = sample_configs[mask]
    sample_energies = sample_energies[mask]
    sample_foms     = sample_foms[mask]

    # step3: sort
    sort_indices    = np.argsort(sample_foms)[::-1]    # Sort by FoM in descending order
    sample_configs  = sample_configs[sort_indices]
    sample_energies = sample_energies[sort_indices]
    sample_foms     = sample_foms[sort_indices]

    # step4: add
    configs  = np.r_[configs, sample_configs]
    energies = np.r_[energies, sample_energies]
    foms     = np.r_[foms, sample_foms]

    return configs, energies, foms

## FMQA

### Generate Initial Data

In [13]:
# generate initial data
if os.path.exists(init_configs_path):
    configs = np.load(init_configs_path)
else:
    configs = generate_unique_binary_array(init_dataset_size=init_dataset_size, rings=rings)
    np.save(init_configs_path, configs)

if os.path.exists(init_foms_path):
    foms = np.load(init_foms_path)
else:
    foms = np.array([
               zp.set_config(config) or zp.figure_of_merit_calc.calc_fom(target)
               for config in configs
           ])
    np.save(init_foms_path, foms)
print(np.min(foms))

0.06323153874498597


### Run

In [14]:
# initialization for running FMQA

actual_new_data = []
energies = np.array([])
min_fom = np.min(foms)
min_fom_iter = 0

# initialize FM
model = FactorizationMachine(input_size=input_size, factorization_size=K, act="identity")
model.init_params(initializer=mx.init.Normal())

In [15]:
for i in range(iteration):

    print(f"*** {i+1}th iteration ***")

    # define file names
    start_time = time.time()
    iter_model_path = os.path.join(iter_model_Q_path, f"{i+1}th_{iter_model_file_name}")
    iter_Q_arr_path = os.path.join(iter_model_Q_path, f"{i+1}th_{iter_Q_arr_file_name}")
    iter_Q_fig_path = os.path.join(iter_model_Q_path, f"{i+1}th_{iter_Q_fig_file_name}")
    end_time = time.time()
    print(f"設定路徑: {end_time - start_time} s")

    # train FM
    start_time = time.time()
    model.train(configs, foms, num_epoch, learning_rate)
    model.save_model(path=iter_model_path)
    end_time = time.time()
    print(f"訓練FM: {end_time - start_time} s")

    # plot Q matrix
    start_time = time.time()
    model.plot_Q_matrix(show_fig=False, path_fig=iter_Q_fig_path, save_fig=True, path_data=iter_Q_arr_path, save_data=True)
    end_time = time.time()
    print(f"畫Q matrix: {end_time - start_time} s")

    # sampling from a QUBO sampler
    start_time = time.time()
    sampleset = sampling(sampler_type=sampler_type, bqm=model.bqm(), num_reads=num_reads)
    end_time = time.time()
    print(f"sampling: {end_time - start_time} s")

    # adding back new data to dataset
    start_time = time.time()
    sample_configs  = sampleset.record["sample"]
    sample_energies = sampleset.record["energy"]
    configs, energies, foms = add_data(configs, energies, foms, sample_configs, sample_energies, adding_method, adding_num)
    end_time = time.time()
    print(f"篩選數據: {end_time - start_time} s")

    # record min FoM
    if min_fom > np.min(foms):
        min_fom      = np.min(foms)
        min_fom_iter = i + 1
    print(f"目前最小的FoM: {min_fom}")

    print("=" * 50)

# save: configs, energies, foms
np.save(configs_path, configs)
np.save(energies_path, energies)
np.save(foms_path, foms)

# save: actual_new_data
actual_new_data = np.array(actual_new_data)
np.save(actual_new_data_path, actual_new_data)

# create a folder with name including "in which iter was the min FoM found" and "min FoM"
min_fom_folder = f"第{min_fom_iter}次iter找到最小FoM:{min_fom}"
min_fom_path = os.path.join(base_path, min_fom_folder)
os.makedirs(min_fom_path)

*** 1th iteration ***
設定路徑: 2.0503997802734375e-05 s
訓練FM: 1.4965884685516357 s
畫Q matrix: 0.479947566986084 s
sampling: 0.041736602783203125 s
篩選數據: 0.02892899513244629 s
目前最小的FoM: 0.06323153874498597
*** 2th iteration ***
設定路徑: 1.9550323486328125e-05 s
訓練FM: 1.4852981567382812 s
畫Q matrix: 0.29074668884277344 s
sampling: 0.04468250274658203 s
篩選數據: 0.027611494064331055 s
目前最小的FoM: 0.06323153874498597
*** 3th iteration ***
設定路徑: 2.384185791015625e-05 s
訓練FM: 1.4571764469146729 s
畫Q matrix: 0.27819013595581055 s
sampling: 0.04242253303527832 s
篩選數據: 0.027211666107177734 s
目前最小的FoM: 0.05470743830167539
*** 4th iteration ***
設定路徑: 2.1696090698242188e-05 s
訓練FM: 1.4784188270568848 s
畫Q matrix: 0.2840607166290283 s
sampling: 0.039391517639160156 s
篩選數據: 0.02554607391357422 s
目前最小的FoM: 0.05470743830167539
*** 5th iteration ***
設定路徑: 2.09808349609375e-05 s
訓練FM: 2.5712356567382812 s
畫Q matrix: 0.47385334968566895 s
sampling: 0.04542422294616699 s
篩選數據: 0.04154706001281738 s
目前最小的FoM: 0.05470